# Proyecto Integrador — Sismos en Perú
**Curso:** Lenguaje de Programación II
**Tema:** Análisis de la actividad sísmica reciente en Perú usando la API pública del USGS

**Requisitos obligatorios cubiertos en este notebook:**
1. Programación Orientada a Objetos (3 clases: `ExtractorSismos`, `ProcesadorSismos`, `VisualizadorSismos`)
2. Programas en red (uso de `requests` para HTTP)
3. Expresiones regulares (extracción de patrón en el texto del lugar del sismo)
4. API con información real y actualizada (USGS Earthquake API)
5. Procesamiento con pandas (limpieza, transformación, análisis)
6. Visualización (3 gráficos distintos)


## 2. Clase `ExtractorSismos` (Programas en red / API)

Esta clase se conecta a la **API pública del USGS** (`https://earthquake.usgs.gov`), que es
gratuita, no requiere API key y entrega datos reales y actualizados de sismos en formato GeoJSON.
Se filtra solo el territorio peruano usando un *bounding box* de coordenadas.
Aquí se cumple el requisito de **Programas en red** (uso de `requests` para HTTP GET).

In [ ]:
# APORTE INTEGRANTE 2: Clase ExtractorSismos (conexión de red)
class ExtractorSismos:
    """Clase encargada de la conexión de red y obtención de datos de sismos en Perú."""

    LAT_MIN, LAT_MAX = -18.5, -0.0
    LON_MIN, LON_MAX = -81.5, -68.5

    def __init__(self, dias_atras: int = 90, magnitud_minima: float = 2.5):
        self.url_base = "https://earthquake.usgs.gov/fdsnws/event/1/query"
        self.dias_atras = dias_atras
        self.magnitud_minima = magnitud_minima
        self.datos_brutos = None

    def conectar_y_descargar(self) -> dict:
        """Realiza una solicitud HTTP GET a la API del USGS y descarga los sismos en Perú."""
        fecha_fin = datetime.utcnow()
        fecha_inicio = fecha_fin - timedelta(days=self.dias_atras)

        parametros = {
            "format": "geojson",
            "starttime": fecha_inicio.strftime("%Y-%m-%d"),
            "endtime": fecha_fin.strftime("%Y-%m-%d"),
            "minlatitude": self.LAT_MIN,
            "maxlatitude": self.LAT_MAX,
            "minlongitude": self.LON_MIN,
            "maxlongitude": self.LON_MAX,
            "minmagnitude": self.magnitud_minima,
            "orderby": "time",
        }

        try:
            print(f"[INFO] Conectando a la red: {self.url_base}")
            respuesta = requests.get(self.url_base, params=parametros, timeout=15)

            if respuesta.status_code == 200:
                self.datos_brutos = respuesta.json()
                total = len(self.datos_brutos.get("features", []))
                print(f"[INFO] Datos descargados correctamente. Sismos encontrados: {total}")
                return self.datos_brutos
            else:
                print(f"[ERROR] Código de estado HTTP inesperado: {respuesta.status_code}")
                return {}
        except requests.exceptions.RequestException as e:
            print(f"[CRÍTICO] Fallo en la conexión de red: {e}")
            return {}

# Ejecutamos la descarga real
extractor = ExtractorSismos(dias_atras=DIAS_A_CONSULTAR, magnitud_minima=MAGNITUD_MINIMA)
datos_crudos = extractor.conectar_y_descargar()
